In [66]:
import pandas as pd
import duckdb
import numpy as np

# Get positive patient counts

In [ ]:
cohort_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Get the oncology cohort (all active and past cancer patients)
diagnoses_sql_queries = ["active_cancer", "personal_history_cancer", "history_and_active"]
for query in diagnoses_sql_queries:
    with open(f"exploration_sql_files/diagnoses_sql/{query}.sql", "r") as f:
        sql: str = f.read()
        cohort_con.execute(sql).df()


oncology_sql_queries = ["prescriptions_sql/prescriptions_count_regex.sql", # Fetch the prescriptions/oncology drugs
                        "cohort_outcomes_sql/first_drug.sql", # Get the first occurrence of drugs
                        "cohort_outcomes_sql/cardiovascular_outcomes.sql", # Get the cardiovascular, ICD outcomes
                        "cohort_outcomes_sql/lvef_outcomes.sql", # Get the LVEF outcomes
                        ]

for query in oncology_sql_queries:
    with open(f"exploration_sql_files/{query}.sql", "r") as f:
        sql: str = f.read()
        cohort_con.execute(sql).df()

# Create the entire cohort:
with open("exploration_sql_files/cohort_outcomes_sql/create_cohorts.sql", "r") as f:
    sql: str = f.read()
    cohort_con.execute(sql)  # just run the CREATE VIEW
    cohort_con_result = cohort_con.execute("SELECT * FROM cardiotoxicity_master").df()

cohort_con_result # takes around 1 minute

,subject_id,first_oncology_time,cv_event_1yr,has_baseline_lvef,has_followup_lvef_1yr,baseline_time,baseline_lvef,first_followup_lvef_time_1yr,worst_followup_lvef_1yr,lvef_drop_10_1yr,lvef_below_50_1yr,definite_lvef_ctrcd_1yr,pre_existing_cv_history,cardiotoxicity_1yr,has_outcome_evidence_1yr,cardiotoxicity_label_1yr,eligible_for_binary_label_1yr
0,11985277,2110-05-19 15:00:00,0,1,1,2110-05-13 16:20:00,58.0,2110-06-23 09:35:00,55.0,0,0,0,0,0,1,negative_observed,1
1,11988400,2172-08-01 15:00:00,0,1,1,2172-07-07 09:14:00,65.0,2172-10-02 11:20:00,60.0,0,0,0,0,0,1,negative_observed,1
2,13693197,2158-05-24 21:00:00,0,1,1,2158-05-21 09:12:00,65.0,2158-08-23 09:00:00,60.0,0,0,0,0,0,1,negative_observed,1
3,13922124,2148-03-31 15:00:00,0,0,1,NaT,NaN,2148-03-31 15:02:00,55.0,0,0,0,0,0,1,negative_observed,1
4,15593172,2199-12-31 12:00:00,0,1,1,2199-06-03 14:05:00,55.0,2200-08-11 09:24:00,50.0,0,0,0,0,0,1,negative_observed,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2540,12373117,2112-03-31 14:00:00,0,1,0,2111-11-05 11:02:00,70.0,NaT,NaN,0,0,0,1,0,0,unknown_no_followup_evidence,0
2541,15794853,2188-09-01 17:00:00,0,0,0,NaT,NaN,NaT,NaN,0,0,0,1,0,0,unknown_no_followup_evidence,0
2542,15999159,2200-02-23 18:00:00,0,0,0,NaT,NaN,NaT,NaN,0,0,0,1,0,0,unknown_no_followup_evidence,0
2543,18471018,2167-12-27 13:00:00,0,1,0,2167-01-11 13:26:00,50.0,NaT,NaN,0,0,0,1,0,0,unknown_no_followup_evidence,0


In [ ]:
cardiotoxicity_df = cohort_con_result.copy()

# Get summary statistics for cardiotoxicity patients

In [77]:
import pandas as pd
import numpy as np


def summarize_cardiotoxicity_cohort(
    cardiotoxicity_df: pd.DataFrame,
    exclude_cv_history: bool = False,
) -> dict[str, pd.DataFrame]:
    """
    Generate summary statistics for the cardiotoxicity cohort.

    Label definitions:
        positive = cardiotoxicity_1yr == 1
        negative = cardiotoxicity_1yr == 0 AND has_outcome_evidence_1yr == 1
        unknown  = has_outcome_evidence_1yr == 0

    Args:
        cardiotoxicity_df:
            DataFrame from cardiotoxicity_master.

        exclude_cv_history:
            If True, exclude patients with pre_existing_cv_history == 1.

    Returns:
        Dictionary of summary tables.
    """

    df = cardiotoxicity_df.copy()

    # -----------------------------
    # 1. Ensure expected columns exist
    # -----------------------------
    required_cols = [
        "subject_id",
        "first_oncology_time",
        "cv_event_1yr",
        "has_baseline_lvef",
        "has_followup_lvef_1yr",
        "baseline_time",
        "baseline_lvef",
        "first_followup_lvef_time_1yr",
        "worst_followup_lvef_1yr",
        "lvef_drop_10_1yr",
        "lvef_below_50_1yr",
        "definite_lvef_ctrcd_1yr",
        "pre_existing_cv_history",
        "cardiotoxicity_1yr",
        "has_outcome_evidence_1yr",
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing expected columns: {missing_cols}")

    # -----------------------------
    # 2. Clean data types
    # -----------------------------
    flag_cols = [
        "cv_event_1yr",
        "has_baseline_lvef",
        "has_followup_lvef_1yr",
        "lvef_drop_10_1yr",
        "lvef_below_50_1yr",
        "definite_lvef_ctrcd_1yr",
        "pre_existing_cv_history",
        "cardiotoxicity_1yr",
        "has_outcome_evidence_1yr",
    ]

    for col in flag_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    date_cols = [
        "first_oncology_time",
        "baseline_time",
        "first_followup_lvef_time_1yr",
    ]

    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    numeric_cols = [
        "baseline_lvef",
        "worst_followup_lvef_1yr",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # -----------------------------
    # 3. Optional exclusion
    # -----------------------------
    original_n = len(df)

    if exclude_cv_history:
        df = df[df["pre_existing_cv_history"] == 0].copy()

    analyzed_n = len(df)

    # -----------------------------
    # 4. Create label column
    # -----------------------------
    conditions = [
        df["cardiotoxicity_1yr"].eq(1),
        df["cardiotoxicity_1yr"].eq(0) & df["has_outcome_evidence_1yr"].eq(1),
        df["has_outcome_evidence_1yr"].eq(0),
    ]

    choices = ["positive", "negative", "unknown"]

    df["label"] = np.select(
        conditions,
        choices,
        default="unknown",
    )

    # Optional ML-friendly numeric label:
    # positive = 1, negative = 0, unknown = -1
    df["label_id"] = df["label"].map(
        {
            "positive": 1,
            "negative": 0,
            "unknown": -1,
        }
    )

    # -----------------------------
    # Helper functions
    # -----------------------------
    def count_pct(series: pd.Series) -> pd.DataFrame:
        counts = series.value_counts(dropna=False)
        out = counts.rename("n").reset_index()
        out.columns = [series.name or "category", "n"]
        out["pct"] = (out["n"] / len(series) * 100).round(2)
        return out

    def binary_flag_summary(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
        rows = []

        for col in columns:
            n_positive = int(data[col].sum())
            n_total = len(data)

            rows.append(
                {
                    "variable": col,
                    "n_positive": n_positive,
                    "n_total": n_total,
                    "pct_positive": round(n_positive / n_total * 100, 2)
                    if n_total > 0
                    else np.nan,
                }
            )

        return pd.DataFrame(rows)

    # -----------------------------
    # 5. Cohort overview
    # -----------------------------
    cohort_overview = pd.DataFrame(
        [
            {
                "metric": "original_n_patients",
                "value": original_n,
            },
            {
                "metric": "analyzed_n_patients",
                "value": analyzed_n,
            },
            {
                "metric": "excluded_due_to_pre_existing_cv_history",
                "value": original_n - analyzed_n,
            },
            {
                "metric": "excluded_cv_history_applied",
                "value": exclude_cv_history,
            },
            {
                "metric": "unique_subjects",
                "value": df["subject_id"].nunique(),
            },
        ]
    )

    # -----------------------------
    # 6. Label breakdown
    # -----------------------------
    label_order = ["positive", "negative", "unknown"]

    label_breakdown = (
        df["label"]
        .value_counts()
        .reindex(label_order, fill_value=0)
        .rename_axis("label")
        .reset_index(name="n")
    )

    label_breakdown["pct"] = (
        label_breakdown["n"] / len(df) * 100
    ).round(2)

    # Label breakdown by CV history, useful before exclusion
    label_by_cv_history = (
        df.groupby(["pre_existing_cv_history", "label"])
        .size()
        .reset_index(name="n")
    )

    label_by_cv_history["pct_within_cv_history_group"] = (
        label_by_cv_history["n"]
        / label_by_cv_history.groupby("pre_existing_cv_history")["n"].transform("sum")
        * 100
    ).round(2)

    # -----------------------------
    # 7. Outcome/component summaries
    # -----------------------------
    outcome_flags = [
        "cv_event_1yr",
        "has_baseline_lvef",
        "has_followup_lvef_1yr",
        "lvef_drop_10_1yr",
        "lvef_below_50_1yr",
        "definite_lvef_ctrcd_1yr",
        "pre_existing_cv_history",
        "cardiotoxicity_1yr",
        "has_outcome_evidence_1yr",
    ]

    outcome_summary = binary_flag_summary(df, outcome_flags)

    # -----------------------------
    # 8. LVEF numeric summaries
    # -----------------------------
    lvef_summary = (
        df[["baseline_lvef", "worst_followup_lvef_1yr"]]
        .describe()
        .T
        .reset_index()
        .rename(columns={"index": "variable"})
    )

    # Add missingness to numeric summary
    lvef_missing = (
        df[["baseline_lvef", "worst_followup_lvef_1yr"]]
        .isna()
        .sum()
        .rename("n_missing")
        .reset_index()
        .rename(columns={"index": "variable"})
    )

    lvef_summary = lvef_summary.merge(lvef_missing, on="variable", how="left")
    lvef_summary["pct_missing"] = (
        lvef_summary["n_missing"] / len(df) * 100
    ).round(2)

    # -----------------------------
    # 9. Time-window summaries
    # -----------------------------
    df["days_baseline_to_drug"] = (
        df["first_oncology_time"] - df["baseline_time"]
    ).dt.days

    df["days_drug_to_first_followup_lvef"] = (
        df["first_followup_lvef_time_1yr"] - df["first_oncology_time"]
    ).dt.days

    time_summary = (
        df[["days_baseline_to_drug", "days_drug_to_first_followup_lvef"]]
        .describe()
        .T
        .reset_index()
        .rename(columns={"index": "variable"})
    )

    time_missing = (
        df[["days_baseline_to_drug", "days_drug_to_first_followup_lvef"]]
        .isna()
        .sum()
        .rename("n_missing")
        .reset_index()
        .rename(columns={"index": "variable"})
    )

    time_summary = time_summary.merge(time_missing, on="variable", how="left")
    time_summary["pct_missing"] = (
        time_summary["n_missing"] / len(df) * 100
    ).round(2)

    # -----------------------------
    # 10. Label-specific outcome summary
    # -----------------------------
    label_component_summary = (
        df.groupby("label")[outcome_flags]
        .mean()
        .mul(100)
        .round(2)
        .reset_index()
    )

    # These are percentages, so rename columns clearly
    label_component_summary = label_component_summary.rename(
        columns={
            col: f"pct_{col}" for col in outcome_flags
        }
    )

    # -----------------------------
    # 11. Return everything
    # -----------------------------
    return {
        "cohort_overview": cohort_overview,
        "label_breakdown": label_breakdown,
        "label_by_cv_history": label_by_cv_history,
        "outcome_summary": outcome_summary,
        "lvef_summary": lvef_summary,
        "time_summary": time_summary,
        "label_component_summary": label_component_summary,
        "analysis_dataframe": df,
    }



def compare_label_breakdowns(
    full_summary: dict[str, pd.DataFrame],
    no_cv_history_summary: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    full = full_summary["label_breakdown"].copy()
    full["cohort"] = "full_cohort"

    no_cv = no_cv_history_summary["label_breakdown"].copy()
    no_cv["cohort"] = "excluded_pre_existing_cv_history"

    comparison = pd.concat([full, no_cv], ignore_index=True)

    comparison = comparison[
        ["cohort", "label", "n", "pct"]
    ].sort_values(["cohort", "label"])

    return comparison

In [78]:
full_summary = summarize_cardiotoxicity_cohort(
    cardiotoxicity_df,
    exclude_cv_history=False,
)

no_cv_history_summary = summarize_cardiotoxicity_cohort(
    cardiotoxicity_df,
    exclude_cv_history=True,
)

display(full_summary["cohort_overview"])
display(full_summary["label_breakdown"])
display(full_summary["outcome_summary"])
display(full_summary["lvef_summary"])
display(full_summary["time_summary"])

display(no_cv_history_summary["cohort_overview"])
display(no_cv_history_summary["label_breakdown"])
display(no_cv_history_summary["outcome_summary"])

display(
    compare_label_breakdowns(
        full_summary,
        no_cv_history_summary,
    )
)

,metric,value
0,original_n_patients,2545
1,analyzed_n_patients,2545
2,excluded_due_to_pre_existing_cv_history,0
3,excluded_cv_history_applied,False
4,unique_subjects,2545


,label,n,pct
0,positive,399,15.68
1,negative,1710,67.19
2,unknown,436,17.13


,variable,n_positive,n_total,pct_positive
0,cv_event_1yr,367,2545,14.42
1,has_baseline_lvef,1231,2545,48.37
2,has_followup_lvef_1yr,1169,2545,45.93
3,lvef_drop_10_1yr,256,2545,10.06
4,lvef_below_50_1yr,177,2545,6.95
5,definite_lvef_ctrcd_1yr,95,2545,3.73
6,pre_existing_cv_history,415,2545,16.31
7,cardiotoxicity_1yr,399,2545,15.68
8,has_outcome_evidence_1yr,2109,2545,82.87


,variable,count,mean,std,min,25%,50%,75%,max,n_missing,pct_missing
0,baseline_lvef,1231.0,60.579204,8.143731,20.0,55.0,60.0,65.0,81.0,1314,51.63
1,worst_followup_lvef_1yr,1169.0,55.561163,10.984963,10.0,50.0,55.0,60.0,80.0,1376,54.07


,variable,count,mean,std,min,25%,50%,75%,max,n_missing,pct_missing
0,days_baseline_to_drug,1231.0,24.648253,58.434486,0.0,1.0,4.0,12.0,364.0,1314,51.63
1,days_drug_to_first_followup_lvef,1169.0,62.372968,72.326039,0.0,12.0,36.0,85.0,345.0,1376,54.07


,metric,value
0,original_n_patients,2545
1,analyzed_n_patients,2130
2,excluded_due_to_pre_existing_cv_history,415
3,excluded_cv_history_applied,True
4,unique_subjects,2130


,label,n,pct
0,positive,184,8.64
1,negative,1597,74.98
2,unknown,349,16.38


,variable,n_positive,n_total,pct_positive
0,cv_event_1yr,162,2130,7.61
1,has_baseline_lvef,970,2130,45.54
2,has_followup_lvef_1yr,938,2130,44.04
3,lvef_drop_10_1yr,192,2130,9.01
4,lvef_below_50_1yr,95,2130,4.46
5,definite_lvef_ctrcd_1yr,57,2130,2.68
6,pre_existing_cv_history,0,2130,0.00
7,cardiotoxicity_1yr,184,2130,8.64
8,has_outcome_evidence_1yr,1781,2130,83.62


,cohort,label,n,pct
4,excluded_pre_existing_cv_history,negative,1597,74.98
3,excluded_pre_existing_cv_history,positive,184,8.64
5,excluded_pre_existing_cv_history,unknown,349,16.38
1,full_cohort,negative,1710,67.19
0,full_cohort,positive,399,15.68
2,full_cohort,unknown,436,17.13
